In [1]:
import pandas as pd

In [2]:
pd.__version__

'3.0.3'

In [ ]:

df = pd.read_pickle('../data/LSWMD_slimmed.pkl')
df.info()



<class 'pandas.DataFrame'>
Index: 12822 entries, 19 to 811447
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   waferMap  12822 non-null  object
 1   label     12822 non-null  str   
dtypes: object(1), str(1)
memory usage: 300.5+ KB


,waferMap,label
19,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc
36,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc
39,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc
40,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc
41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc


In [21]:
import tensorflow as tf
import numpy as np



TARGET_SIZE = (96, 96) 

def wafertoscale(wafer_matrix): #preprocesses single wafermap
   
    tensor = tf.convert_to_tensor(wafer_matrix, dtype=tf.float32)  
    tensor = tf.expand_dims(tensor, axis=-1) # Add the channel dimension: (Height, Width) -> (Height, Width, 1)

    # Resize with pad:
    resized_tensor = tf.image.resize_with_pad(
        tensor, 
        target_height=TARGET_SIZE[0], 
        target_width=TARGET_SIZE[1], 
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )
    #print(tensor.shape,resized_tensor.shape)

    return resized_tensor.numpy()

# Apply preprocessing to all wafers
df['processed_waferMap'] = [wafertoscale(w) for w in df['waferMap']]
X_processed = np.stack([wafertoscale(w) for w in df['waferMap']])

print(f"Final Image Data Shape: {X_processed.shape}") 

df.head()


Final Image Data Shape: (12822, 96, 96, 1)


,waferMap,label,processed_waferMap
19,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc,"[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0..."
36,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc,"[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0..."
39,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc,"[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0..."
40,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc,"[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0..."
41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc,"[[[0.0], [0.0], [0.0], [0.0], [0.0], [0.0], [0..."


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
# Convert string labels to integers do test train split
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['label'])

num_classes = len(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)


X_train = X_train.astype(np.int8)
X_test = X_test.astype(np.int8)

# save processed wafers to a compressed file
np.savez_compressed(
    'wafer_dataset_96x96.npz', 
    X_train=X_train, 
    y_train=y_train, 
    X_test=X_test, 
    y_test=y_test
)

In [23]:
# Load processed wafers
loaded_data = np.load('wafer_dataset_96x96.npz')

# Extract the arrays
X_train = loaded_data['X_train']
y_train = loaded_data['y_train']
X_test = loaded_data['X_test']
y_test = loaded_data['y_test']

# Re-wrap into tf.data.Dataset instantly
BATCH_SIZE = 32
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train)).batch(BATCH_SIZE)

BATCH_SIZE = 32

# Create Training Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = (
    train_dataset
    .shuffle(buffer_size=len(X_train)) # Shuffle the training data
    .batch(BATCH_SIZE)                 # Group into batches of 32
    .prefetch(tf.data.AUTOTUNE)        # Pre-fetch next batch in the background
)

# Create Testing/Validation Dataset
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)                 
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# CNN model
model = tf.keras.models.Sequential([
    # ---- Conv block 1 -------------------------------------------------
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu',
                           input_shape=(96, 96, 1)),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Conv block 2 -------------------------------------------------
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Conv block 3 -------------------------------------------------
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.25),

    # ---- Flatten + Dense -----------------------------------------------
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    ])

model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics = ['accuracy']
)

model.summary()